# Lab05 - Streams

In [2]:
import numpy as np
import numba
from numba import cuda
import time
import math

cuda.detect()

Found 1 CUDA devices
id 0    b'NVIDIA GeForce RTX 2060'                              [SUPPORTED]
                      Compute Capability: 7.5
                           PCI Device ID: 0
                              PCI Bus ID: 1
                                    UUID: GPU-006291b6-94bd-bac8-588d-4a97f26284e1
                                Watchdog: Enabled
                            Compute Mode: WDDM
             FP32/FP64 Performance Ratio: 32
Summary:
	1/1 devices are supported


True

## Ex. 1) Matrix Multiplication w/ SMEM and Pinned Memory

In [ ]:
# With SMEM:

SIZE_A = (1027, 2047)
SIZE_B = (2047, 1248)
SIZE_C = (SIZE_A[0], SIZE_B[1])

BLOCK_SIZE = (32, 32)
GRID_SIZE = ((SIZE_C[1] + BLOCK_SIZE[0] - 1) // BLOCK_SIZE[0], (SIZE_C[0] + BLOCK_SIZE[1] - 1) // BLOCK_SIZE[1] )

print(f"Grid Size: {GRID_SIZE}")
print(f"Block Size: {BLOCK_SIZE}")

A_host = np.random.randint(0, 10, SIZE_A)
B_host = np.random.randint(0, 10, SIZE_B)
C_host = np.zeros(SIZE_C)

# This is the new part:

A_pinned = cuda.pinned_array_like(A_host)
B_pinned = cuda.pinned_array_like(B_host)
C_pinned = cuda.pinned_array_like(C_host)

A_pinned[:] = A_host
B_pinned[:] = B_host

A_device = cuda.to_device(A_pinned)
B_device = cuda.to_device(B_pinned)
C_device = cuda.device_array_like(C_pinned)

@cuda.jit
def mat_mul(A, B, C):
    global_idx_col, global_idx_row = cuda.grid(2)
    thread_idx_col, thread_idx_row = cuda.threadIdx.x, cuda.threadIdx.y
    
    # Notice that we remove the early return from here, cause those threads should still 
    # contribute copying into the SMEM, not return instantly!
    
    # Each block allocates room for a tile from A and a tile from B
    # The tiles are of the size of the block, as we want 1 thread <-> 1 value in C
    smem_A = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    smem_B = cuda.shared.array(BLOCK_SIZE, dtype=numba.float32)
    
    # Now we want to copy different tiles into smem_A and smem_B
    tiles_count = (A.shape[1] + BLOCK_SIZE[0] - 1) // BLOCK_SIZE[0] 
    acc = 0.0
    for tile in range(tiles_count):
        col_A = BLOCK_SIZE[0] * tile + thread_idx_col
        row_B = BLOCK_SIZE[0] * tile + thread_idx_row
        
        # 1st: each thread copies one value from A and one value from B
        # this copies the two tiles from A and B into smem_A and smem_B
        if global_idx_row < A.shape[0] and col_A < A.shape[1]:
            smem_A[thread_idx_row, thread_idx_col] = A[global_idx_row, col_A]
        else:
            smem_A[thread_idx_row, thread_idx_col] = 0.0
            
        if row_B < B.shape[0] and global_idx_col < B.shape[1]:
            smem_B[thread_idx_row, thread_idx_col] = B[row_B, global_idx_col]
        else:
            smem_B[thread_idx_row, thread_idx_col] = 0.0

        cuda.syncthreads()
        
        # 2nd: it accumulates into acc the partial results from the two tiles  
        for i in range(BLOCK_SIZE[0]):
            acc += smem_A[thread_idx_row, i] * smem_B[i, thread_idx_col]
        
        cuda.syncthreads()
    
    if global_idx_row < C.shape[0] and global_idx_col < C.shape[1]:
        C[global_idx_row, global_idx_col] = acc
   
mat_mul[GRID_SIZE, BLOCK_SIZE](A_device, B_device, C_device)
cuda.synchronize()
 
expected = A_host @ B_host

C_pinned = C_device.copy_to_host() # Notice here as well!!

print(f"Expected: {expected}")
print(f"Got: {C_pinned}")

Grid Size: (39, 33)
Block Size: (32, 32)
Expected: [[42684 40909 41379 ... 41136 41465 40378]
 [41190 41063 41197 ... 40288 40729 40305]
 [42705 41809 41557 ... 42254 42465 40985]
 ...
 [42671 41676 42222 ... 42308 41506 41113]
 [40763 40099 39823 ... 39971 39741 39580]
 [42709 41933 41913 ... 41595 41606 41237]]
Got: [[42684. 40909. 41379. ... 41136. 41465. 40378.]
 [41190. 41063. 41197. ... 40288. 40729. 40305.]
 [42705. 41809. 41557. ... 42254. 42465. 40985.]
 ...
 [42671. 41676. 42222. ... 42308. 41506. 41113.]
 [40763. 40099. 39823. ... 39971. 39741. 39580.]
 [42709. 41933. 41913. ... 41595. 41606. 41237.]]


## Ex. 2) Function Tabulation w/ Streams

Given a 1D array of x values, compute the y values w.r.t. a certain function

In [ ]:
# Default Stream

@cuda.jit(device=True, inline=True)
def f(x):
    s = math.sin(x)
    c = math.cos(x)
    return math.sqrt(abs(s * s - c * c))


SIZE = 64 * 1024 * 1024

BLOCK_SIZE = 32
GRID_SIZE = (SIZE + BLOCK_SIZE - 1) // BLOCK_SIZE

x_host = np.linspace(0, 2*math.pi, SIZE, dtype=np.float32)
y_host = np.zeros(SIZE)

x_pinned = cuda.pinned_array_like(x_host)
y_pinned = cuda.pinned_array_like(y_host)

x_pinned[:] = x_host

x_device = cuda.to_device(x_pinned)
y_device = cuda.device_array_like(y_pinned)

@cuda.jit
def tabulate(x, y):
    global_idx = cuda.grid(1)
    if global_idx < SIZE:
        y[global_idx] = f(x[global_idx])


tabulate[GRID_SIZE, BLOCK_SIZE](x_device, y_device)
cuda.synchronize()

y_pinned = y_device.copy_to_host()

print(x_pinned)
print(y_pinned)

[0.0000000e+00 9.3626760e-08 1.8725352e-07 ... 6.2831850e+00 6.2831850e+00
 6.2831855e+00]
[1. 1. 1. ... 1. 1. 1.]


In [10]:
# Multiple Streams

@cuda.jit(device=True, inline=True)
def f(x):
    s = math.sin(x)
    c = math.cos(x)
    return math.sqrt(abs(s * s - c * c))


SIZE = 64 * 1024 * 1024

STREAMS = 4
STREAM_CHUNK = (SIZE + STREAMS - 1) // STREAMS

BLOCK_SIZE = 32
GRID_SIZE = (STREAM_CHUNK + BLOCK_SIZE - 1) // BLOCK_SIZE


x_host = np.linspace(0, 2*math.pi, SIZE, dtype=np.float64)
y_host = np.zeros(SIZE)

x_pinned = cuda.pinned_array_like(x_host)
y_pinned = cuda.pinned_array_like(y_host)

x_pinned[:] = x_host

@cuda.jit
def tabulate(x, y):
    global_idx = cuda.grid(1)
    if global_idx < len(x):
        y[global_idx] = f(x[global_idx])

# 1: Create the Streams
streams = list()
for _ in range(STREAMS):
    streams.append(cuda.stream())

# 2: Use the Streams
# Each Stream will first copy to device and then compute

for stream_idx, stream in enumerate(streams):
    left_idx = stream_idx * STREAM_CHUNK
    right_idx = min((stream_idx + 1) * STREAM_CHUNK, SIZE)

    x_device = cuda.to_device(x_pinned[left_idx : right_idx], stream=stream)
    y_device = cuda.device_array_like(x_device)

    tabulate[GRID_SIZE, BLOCK_SIZE, stream](x_device, y_device)
    # No synchronize!!

    y_device.copy_to_host(
        y_pinned[left_idx : right_idx],
        stream=stream
    )
    
# 3: Sync the Streams
for stream in streams:
    stream.synchronize()

print(x_pinned)
print(y_pinned)

[0.00000000e+00 9.36267585e-08 1.87253517e-07 ... 6.28318512e+00
 6.28318521e+00 6.28318531e+00]
[1. 1. 1. ... 1. 1. 1.]


## Ex. 3) Convolution w/ SMEM and Pinned Memory and Streams

In [31]:
# Basically we need to load into SMEM whatever the block needs from arr to compute out
# That is a vector of size BLOCK_SIZE + MASK_SIZE - 1

# About Streams:
# There is a little detail: to each kernel we need to give not only the part of the array that it is going to compute,
# but also the halos for that part.

SIZE = 1024
MASK_RADIUS = 3
MASK_SIZE = 2 * MASK_RADIUS + 1

STREAMS = 4
STREAM_CHUNK = (SIZE + STREAMS - 1) // STREAMS

BLOCK_SIZE = min(32, SIZE)
GRID_SIZE = (STREAM_CHUNK + BLOCK_SIZE - 1) // BLOCK_SIZE # Notice the change here!

SMEM_SIZE = BLOCK_SIZE + MASK_SIZE - 1

arr_host = np.random.randint(0, 10, SIZE)
mask_host = np.random.randint(0, 10, MASK_SIZE)
out_host = np.zeros(SIZE, dtype=np.int32)

print(arr_host)
print(mask_host)

arr_pinned = cuda.pinned_array_like(arr_host)
mask_pinned = cuda.pinned_array_like(mask_host)
out_pinned = cuda.pinned_array_like(out_host)

arr_pinned[:] = arr_host
mask_pinned[:] = mask_host

# We do this only once for all streams
mask_device = cuda.to_device(mask_pinned)

@cuda.jit
def conv_1d(arr, mask, out, halo_offset):
    global_idx = cuda.grid(1)
    thread_idx = cuda.threadIdx.x
    
    # 1st: load data into the SMEM
    smem = cuda.shared.array(SMEM_SIZE, dtype=numba.float32)
    
    # Index into the expanded (halo) arr
    arr_idx = global_idx + halo_offset
    
    # left halo: the first MASK_RADIUS threads copy a piece in here
    if thread_idx < MASK_RADIUS:
        if arr_idx - MASK_RADIUS >= 0:
            smem[thread_idx] = arr[arr_idx - MASK_RADIUS]
        else:
            smem[thread_idx] = 0.0
    
    # easy part: each thread copies one cell in the middle
    if arr_idx < len(arr):
        smem[MASK_RADIUS + thread_idx] = arr[arr_idx]
    else:
        smem[MASK_RADIUS + thread_idx] = 0.0
    
    # right halo: the first MASK_RADIUS threads copy a piece in here
    if thread_idx < MASK_RADIUS:
        if arr_idx + BLOCK_SIZE < len(arr):
            smem[BLOCK_SIZE + MASK_RADIUS + thread_idx] = arr[arr_idx + BLOCK_SIZE]
        else:
            smem[BLOCK_SIZE + MASK_RADIUS + thread_idx] = 0.0
        
    cuda.syncthreads()
    
    if global_idx >= len(out):
        return
    
    mask_start_idx = thread_idx
    mask_end_idx = thread_idx + MASK_SIZE - 1
    
    acc = 0.0
    mask_idx = MASK_SIZE - 1
    for j in range(mask_start_idx, mask_end_idx + 1):
        acc += smem[j] * mask[mask_idx]
        mask_idx -= 1

    out[global_idx] = np.int32(acc)
    
    
# We introduce streams here
    
expected = np.convolve(arr_host, mask_host, mode='same')

streams = list()
for _ in range(STREAMS):
    streams.append(cuda.stream())
    
for stream_idx, stream in enumerate(streams):
    chunk_start_idx = stream_idx * STREAM_CHUNK
    chunk_end_idx = min((stream_idx + 1) * STREAM_CHUNK, SIZE)
    chunk_size = chunk_end_idx - chunk_start_idx
    
    # Important: each stream also needs the halo parts:
    halo_start = max(0, chunk_start_idx - MASK_RADIUS)
    halo_end = min(SIZE, chunk_end_idx + MASK_RADIUS)
    halo_offset = chunk_start_idx - halo_start  # how far into arr_device the real data starts
    
    # H2D
    arr_device = cuda.to_device(
        arr_pinned[halo_start : halo_end],
        stream=stream
    )
    out_device = cuda.device_array(chunk_size, dtype=np.int32)
    
    # Kernel
    conv_1d[GRID_SIZE, BLOCK_SIZE, stream](arr_device, mask_device, out_device, halo_offset)

    # D2H
    out_device.copy_to_host(
        out_pinned[chunk_start_idx : chunk_end_idx],
        stream=stream
    )
    
for stream in streams:
    stream.synchronize()


print(f"Expected: {expected}")
print(f"Got: {out_pinned}")

assert all([a == b for a, b in zip(expected, out_pinned)])


[4 3 1 ... 6 1 7]
[8 0 3 4 0 3 1]
Expected: [ 89  63 104 ...  55  49  51]
Got: [ 89  63 104 ...  55  49  51]


c:\Users\Filippo Corti\Documents\GitHub\GPUComputing\.venv\Lib\site-packages\numba\cuda\dispatcher.py:536: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [ ]:
# 1D Convolution

SIZE = 64 * 1024
KERNEL_RADIUS = 3
KERNEL_SIZE = 2 * KERNEL_RADIUS + 1
BLOCK_SIZE = 64
STREAMS = 8
SIZE_PER_STREAM = (SIZE + STREAMS - 1) // STREAMS
GRID_SIZE = (SIZE_PER_STREAM + BLOCK_SIZE - 1) // BLOCK_SIZE 

SMEM_SIZE = BLOCK_SIZE + 2 * KERNEL_RADIUS

@cuda.jit
def conv(x, k, y):
    global_idx = cuda.grid(1)
    thread_idx = cuda.threadIdx.x
    block_idx = cuda.blockIdx.x
    block_size = cuda.blockDim.x
    first_thread_in_block = block_size * block_idx
    
    smem = cuda.shared.array(SMEM_SIZE, dtype=numba.float32)
    
    # Fill up the SMEM
    if thread_idx < KERNEL_RADIUS:
        if global_idx - KERNEL_RADIUS >= 0:
            smem[thread_idx] = x[global_idx - KERNEL_RADIUS]
        else:
            smem[thread_idx] = 0.0
            
    if thread_idx < BLOCK_SIZE:
        smem[thread_idx + KERNEL_RADIUS] = x[global_idx]
    else:
        smem[thread_idx + KERNEL_RADIUS] = 0.0
    
    if thread_idx >= BLOCK_SIZE - KERNEL_RADIUS:
        if global_idx + KERNEL_RADIUS < x.shape[0]:
            smem[thread_idx + 2 * KERNEL_RADIUS] = x[global_idx + KERNEL_RADIUS]
        else:
            smem[thread_idx + 2 * KERNEL_RADIUS] = 0.0
      
    cuda.syncthreads()      
    
    # Compute Convolution using SMEM
    acc = 0.0
    for delta in range(-KERNEL_RADIUS, KERNEL_RADIUS + 1):
        k_idx = KERNEL_SIZE - 1 - (delta + KERNEL_RADIUS)
        smem_idx = thread_idx + KERNEL_RADIUS + delta
        acc += smem[smem_idx] * k[k_idx]
    
    if global_idx < x.shape[0]:
        y[global_idx] = acc
        
x_host = np.random.rand(SIZE)
k_host = np.random.rand(KERNEL_SIZE)
y_host = np.zeros_like(x_host)

x_pinned = cuda.pinned_array_like(x_host)
k_pinned = cuda.pinned_array_like(k_host)

x_pinned[:] = x_host
k_pinned[:] = k_host

x_device = cuda.device_array(SIZE)
k_device = cuda.to_device(k_pinned)
y_device = cuda.device_array(SIZE)

streams = list()
for stream in range(STREAMS):
    streams.append(cuda.stream())

for stream_idx, stream in enumerate(streams):
    start_idx = stream_idx * SIZE_PER_STREAM
    end_idx = min(SIZE, start_idx + SIZE_PER_STREAM)
    
    # H2D
    x_device[start_idx : end_idx].copy_to_device(
        x_pinned[start_idx : end_idx],
        stream
    )
    
    # Compute
    conv[GRID_SIZE, BLOCK_SIZE, stream](x_device[start_idx : end_idx], k_device, y_device[start_idx : end_idx])
    
    # D2H
    y_device[start_idx : end_idx].copy_to_host(
        y_host[start_idx : end_idx],
        stream
    )
    
for stream in streams:
    stream.synchronize()
    


In [ ]:
# 2D Convolution

KERNEL_RADIUS = 3
KERNEL_SIZE = (KERNEL_RADIUS * 2 + 1, KERNEL_RADIUS * 2 + 1)

@cuda.jit
def conv2d(x, k, y):
    col, row = cuda.grid(2)
    
    if row >= x.shape[0] or col >= x.shape[1]:
        return
    
    acc = 0.0
    for drow in range(-KERNEL_RADIUS, KERNEL_RADIUS + 1):
        nrow = row + drow
        for dcol in range(-KERNEL_RADIUS, KERNEL_RADIUS + 1):
            ncol = col + dcol
            
            if nrow < 0 or nrow >= x.shape[0] or ncol < 0 or ncol >= x.shape[1]:
                continue
            
            acc += x[nrow, ncol] * k[drow + KERNEL_RADIUS, dcol + KERNEL_RADIUS]
            
    y[row, col] = acc
    